# Sequence 2 : Fondamentaux d'algorithmique

Ce notebook couvre les bases algorithmiques necessaires pour le bloc de Recherche Operationnelle : recursivite, strategies de resolution (glouton, diviser pour regner, programmation dynamique, backtracking) et introduction a la notion de complexite.

Ces concepts seront mobilises tout au long des boucles, de la theorie des graphes aux metaheuristiques.

Le notebook est concu pour etre execute de haut en bas : plusieurs cellules reutilisent des variables definies dans les sections precedentes.

## 1. Notion de complexite algorithmique

La complexite mesure la quantite de ressources (temps, memoire) qu'un algorithme consomme en fonction de la taille de l'entree $n$.

On utilise la notation **grand-O** pour decrire le comportement asymptotique :

| Complexite | Nom | Exemple |
|---|---|---|
| $O(1)$ | Constante | Acces a un element de liste par indice |
| $O(\log n)$ | Logarithmique | Recherche dichotomique |
| $O(n)$ | Lineaire | Parcours d'une liste |
| $O(n \log n)$ | Quasi-lineaire | Tri fusion |
| $O(n^2)$ | Quadratique | Tri par insertion (pire cas) |
| $O(2^n)$ | Exponentielle | Enumeration de tous les sous-ensembles |
| $O(n!)$ | Factorielle | Enumeration de toutes les permutations |

Pour le projet ADEME, la distinction entre $O(n^2)$ et $O(2^n)$ est cruciale : le TSP par force brute est en $O(n!)$, ce qui le rend impraticable pour de grandes instances.

In [ ]:
import time
import math


def mesurer_temps(fonction, *args):
    """Mesure le temps CPU d'execution d'une fonction en millisecondes."""
    debut = time.process_time()
    resultat = fonction(*args)
    fin = time.process_time()
    duree_ms = (fin - debut) * 1000
    return resultat, duree_ms


# Illustration : O(n) vs O(n^2)
def somme_lineaire(n):
    """Somme des entiers de 0 a n-1 en O(n)."""
    total = 0
    for i in range(n):
        total += i
    return total


def somme_paires_quadratique(n):
    """Somme de toutes les paires (i+j) en O(n^2)."""
    total = 0
    for i in range(n):
        for j in range(n):
            total += i + j
    return total


for taille in [1000, 5000, 10000]:
    _, t_lin = mesurer_temps(somme_lineaire, taille)
    _, t_quad = mesurer_temps(somme_paires_quadratique, taille)
    print(f"n = {taille:>6} | O(n) : {t_lin:>8.3f} ms | O(n^2) : {t_quad:>8.3f} ms")

## 2. Recursivite

La recursivite est un mecanisme ou une fonction s'appelle elle-meme pour resoudre un probleme en le decomposant en sous-problemes plus petits.

Chaque fonction recursive doit avoir :
- Un **cas de base** : condition d'arret
- Un **cas recursif** : appel a la fonction avec un probleme reduit

In [ ]:
def factorielle(n):
    """Calcul recursif de n!."""
    if n <= 1:
        return 1
    return n * factorielle(n - 1)


def fibonacci(n):
    """Calcul recursif naif du n-ieme nombre de Fibonacci."""
    if n <= 1:
        return n
    return fibonacci(n - 1) + fibonacci(n - 2)


print("10! =", factorielle(10))
print("Fibonacci(10) =", fibonacci(10))

# Attention : fibonacci naif est en O(2^n), c'est tres lent
_, t = mesurer_temps(fibonacci, 30)
print(f"Fibonacci(30) en {t:.1f} ms")

### Visualiser l'explosion combinatoire

Pour comprendre pourquoi certains problemes sont intractables, comparons la croissance de differentes fonctions de complexite.

In [ ]:
print(f"{'n':>4} | {'n^2':>12} | {'2^n':>16} | {'n!':>20}")
print("-" * 60)

for n in [5, 10, 15, 20, 25]:
    n_carre = n ** 2
    deux_puiss_n = 2 ** n
    n_fact = math.factorial(n)
    print(f"{n:>4} | {n_carre:>12} | {deux_puiss_n:>16} | {n_fact:>20}")

print("\nAvec 20 villes, une enumeration exhaustive du TSP")
print(f"necessite d'evaluer {math.factorial(20):,} permutations.")

## 3. Recherche dichotomique

La recherche dichotomique (ou binaire) est l'exemple classique d'un algorithme en $O(\log n)$. Elle s'applique sur une liste **triee** en divisant l'intervalle de recherche par deux a chaque etape.

In [ ]:
def recherche_lineaire(liste, cible):
    """Recherche sequentielle en O(n)."""
    for i, element in enumerate(liste):
        if element == cible:
            return i
    return -1


def recherche_dichotomique(liste_triee, cible):
    """Recherche dichotomique en O(log n)."""
    gauche = 0
    droite = len(liste_triee) - 1

    while gauche <= droite:
        milieu = (gauche + droite) // 2

        if liste_triee[milieu] == cible:
            return milieu
        elif liste_triee[milieu] < cible:
            gauche = milieu + 1
        else:
            droite = milieu - 1

    return -1


# Comparaison sur une grande liste triee
donnees = list(range(1_000_000))
cible = 999_999  # Pire cas pour la recherche lineaire

_, t_lin = mesurer_temps(recherche_lineaire, donnees, cible)
_, t_dicho = mesurer_temps(recherche_dichotomique, donnees, cible)

print(f"Recherche lineaire  : {t_lin:.3f} ms")
print(f"Recherche dichotomique : {t_dicho:.3f} ms")

## 4. Strategie gloutonne

Un algorithme glouton fait a chaque etape le choix localement optimal, sans revenir en arriere. Cette strategie ne garantit pas toujours la solution optimale, mais elle est rapide et fournit souvent une bonne approximation.

### Exemple : le probleme du sac a dos (version continue)

On dispose d'objets avec un poids et une valeur. On veut remplir un sac de capacite limitee pour maximiser la valeur totale. En version continue, on peut prendre une fraction d'objet.

In [ ]:
def sac_a_dos_glouton(objets, capacite):
    """
    Sac a dos fractionnaire (version continue) par strategie gloutonne.

    Parametres
    ----------
    objets : list[tuple[str, float, float]]
        Liste de (nom, poids, valeur).
    capacite : float
        Capacite maximale du sac.

    Retourne
    --------
    list[tuple[str, float]], float
        Les objets selectionnes avec leur fraction, et la valeur totale.
    """
    # Trier par rapport valeur/poids decroissant
    objets_tries = sorted(objets, key=lambda o: o[2] / o[1], reverse=True)

    selection = []
    valeur_totale = 0.0
    poids_restant = capacite

    for nom, poids, valeur in objets_tries:
        if poids_restant <= 0:
            break

        if poids <= poids_restant:
            selection.append((nom, 1.0))
            valeur_totale += valeur
            poids_restant -= poids
        else:
            fraction = poids_restant / poids
            selection.append((nom, round(fraction, 2)))
            valeur_totale += valeur * fraction
            poids_restant = 0

    return selection, round(valeur_totale, 2)


objets = [
    ("Colis_A", 10, 60),
    ("Colis_B", 20, 100),
    ("Colis_C", 30, 120),
]
capacite_max = 50

selection, valeur = sac_a_dos_glouton(objets, capacite_max)
print("Selection :")
for nom, fraction in selection:
    print(f"  {nom} : {fraction * 100:.0f}%")
print(f"Valeur totale : {valeur}")

## 5. Diviser pour regner

Le principe consiste a :
1. **Diviser** le probleme en sous-problemes plus petits
2. **Regner** en resolvant chaque sous-probleme recursivement
3. **Combiner** les solutions des sous-problemes

### Exemple : tri fusion (merge sort)

Le tri fusion est un algorithme classique en $O(n \log n)$.

In [ ]:
def tri_fusion(liste):
    """Tri fusion en O(n log n)."""
    if len(liste) <= 1:
        return liste

    milieu = len(liste) // 2
    gauche = tri_fusion(liste[:milieu])
    droite = tri_fusion(liste[milieu:])

    return fusionner(gauche, droite)


def fusionner(gauche, droite):
    """Fusionne deux listes triees en une seule liste triee."""
    resultat = []
    i = 0
    j = 0

    while i < len(gauche) and j < len(droite):
        if gauche[i] <= droite[j]:
            resultat.append(gauche[i])
            i += 1
        else:
            resultat.append(droite[j])
            j += 1

    resultat.extend(gauche[i:])
    resultat.extend(droite[j:])
    return resultat


import random
random.seed(42)

donnees_aleatoires = [random.randint(0, 1000) for _ in range(20)]
print("Avant tri :", donnees_aleatoires)
print("Apres tri :", tri_fusion(donnees_aleatoires))

In [ ]:
# Comparaison : tri par insertion O(n^2) vs tri fusion O(n log n)

def tri_insertion(liste):
    """Tri par insertion en O(n^2)."""
    resultat = list(liste)
    for i in range(1, len(resultat)):
        cle = resultat[i]
        j = i - 1
        while j >= 0 and resultat[j] > cle:
            resultat[j + 1] = resultat[j]
            j -= 1
        resultat[j + 1] = cle
    return resultat


for taille in [1000, 5000, 10000]:
    donnees = [random.randint(0, 100000) for _ in range(taille)]
    _, t_insert = mesurer_temps(tri_insertion, donnees)
    _, t_fusion = mesurer_temps(tri_fusion, donnees)
    print(f"n = {taille:>6} | Insertion : {t_insert:>8.1f} ms | Fusion : {t_fusion:>8.1f} ms")

## 6. Programmation dynamique

La programmation dynamique resout un probleme en combinant les solutions de sous-problemes **chevauchants**. A la difference du diviser pour regner, chaque sous-probleme n'est resolu qu'une fois, et son resultat est stocke (memoisation).

### Fibonacci avec memoisation

On reprend le calcul de Fibonacci, cette fois en $O(n)$ au lieu de $O(2^n)$.

In [ ]:
def fibonacci_memo(n, memo=None):
    """Fibonacci avec memoisation en O(n)."""
    if memo is None:
        memo = {}

    if n in memo:
        return memo[n]
    if n <= 1:
        return n

    memo[n] = fibonacci_memo(n - 1, memo) + fibonacci_memo(n - 2, memo)
    return memo[n]


def fibonacci_iteratif(n):
    """Fibonacci iteratif (programmation dynamique ascendante) en O(n)."""
    if n <= 1:
        return n

    tableau = [0] * (n + 1)
    tableau[1] = 1

    for i in range(2, n + 1):
        tableau[i] = tableau[i - 1] + tableau[i - 2]

    return tableau[n]


# Comparaison
n_test = 35
_, t_memo = mesurer_temps(fibonacci_memo, n_test)
_, t_iter = mesurer_temps(fibonacci_iteratif, n_test)

print(f"Fibonacci({n_test}) = {fibonacci_iteratif(n_test)}")
print(f"Memoisation : {t_memo:.3f} ms")
print(f"Iteratif    : {t_iter:.3f} ms")

### Sac a dos 0/1 (version entiere)

Contrairement a la version continue, ici on ne peut pas prendre une fraction d'objet. La programmation dynamique permet de resoudre ce probleme en $O(n \times C)$ ou $C$ est la capacite.

In [ ]:
def sac_a_dos_dynamique(objets, capacite):
    """
    Sac a dos 0/1 par programmation dynamique.

    Parametres
    ----------
    objets : list[tuple[str, int, int]]
        Liste de (nom, poids, valeur).
    capacite : int
        Capacite maximale du sac.

    Retourne
    --------
    int, list[str]
        Valeur maximale et liste des objets selectionnes.
    """
    n = len(objets)
    # Tableau de programmation dynamique
    dp = [[0] * (capacite + 1) for _ in range(n + 1)]

    for i in range(1, n + 1):
        nom, poids, valeur = objets[i - 1]
        for c in range(capacite + 1):
            dp[i][c] = dp[i - 1][c]  # On ne prend pas l'objet i
            if poids <= c:
                avec_objet = dp[i - 1][c - poids] + valeur
                dp[i][c] = max(dp[i][c], avec_objet)

    # Reconstruction de la solution
    selection = []
    c = capacite
    for i in range(n, 0, -1):
        if dp[i][c] != dp[i - 1][c]:
            selection.append(objets[i - 1][0])
            c -= objets[i - 1][1]

    return dp[n][capacite], list(reversed(selection))


objets_entiers = [
    ("Colis_A", 10, 60),
    ("Colis_B", 20, 100),
    ("Colis_C", 30, 120),
]

valeur_max, selection = sac_a_dos_dynamique(objets_entiers, 50)
print(f"Valeur maximale : {valeur_max}")
print(f"Objets selectionnes : {selection}")

## 7. Backtracking (retour sur trace)

Le backtracking explore systematiquement toutes les solutions possibles en construisant l'espace de recherche sous forme d'arbre. Des qu'une branche ne peut plus mener a une solution valide, on revient en arriere (**elagage**).

### Exemple : enumeration des permutations

Generer toutes les permutations de $n$ elements est une etape cle pour l'approche par force brute du TSP.

In [ ]:
def generer_permutations(elements):
    """
    Genere toutes les permutations d'une liste par backtracking.

    Retourne une liste de permutations.
    """
    resultats = []

    def backtrack(chemin, restants):
        if not restants:
            resultats.append(list(chemin))
            return

        for i, element in enumerate(restants):
            chemin.append(element)
            backtrack(chemin, restants[:i] + restants[i + 1:])
            chemin.pop()  # Retour sur trace

    backtrack([], elements)
    return resultats


perms = generer_permutations(["A", "B", "C"])
print(f"{len(perms)} permutations de [A, B, C] :")
for p in perms:
    print(" ", p)

### TSP par force brute

On peut utiliser le backtracking pour enumerer toutes les tournees et trouver la tournee optimale. Cette approche est en $O(n!)$, elle n'est viable que pour de petites instances.

In [ ]:
def tsp_force_brute(matrice_dist):
    """
    Resolution du TSP par enumeration exhaustive.

    Retourne la tournee optimale et sa distance.
    """
    n = len(matrice_dist)
    # On fixe le depart a 0 pour eviter les tournees equivalentes
    autres_villes = list(range(1, n))

    meilleure_tournee = None
    meilleure_distance = float("inf")

    def backtrack(chemin, distance_courante, restantes):
        nonlocal meilleure_tournee, meilleure_distance

        # Elagage : si la distance courante depasse deja le meilleur, on arrete
        if distance_courante >= meilleure_distance:
            return

        if not restantes:
            # Ajouter le retour au depart
            distance_totale = distance_courante + matrice_dist[chemin[-1]][0]
            if distance_totale < meilleure_distance:
                meilleure_distance = distance_totale
                meilleure_tournee = list(chemin) + [0]
            return

        for i, ville in enumerate(restantes):
            nouvelle_distance = distance_courante + matrice_dist[chemin[-1]][ville]
            chemin.append(ville)
            backtrack(chemin, nouvelle_distance, restantes[:i] + restantes[i + 1:])
            chemin.pop()

    backtrack([0], 0.0, autres_villes)
    return meilleure_tournee, round(meilleure_distance, 2)


# Test sur un petit graphe (6 villes)
import math
import random
random.seed(42)

nb_villes = 8
coords = [(random.randint(0, 100), random.randint(0, 100)) for _ in range(nb_villes)]

mat = [[0.0] * nb_villes for _ in range(nb_villes)]
for i in range(nb_villes):
    for j in range(i + 1, nb_villes):
        d = math.sqrt((coords[j][0] - coords[i][0]) ** 2 + (coords[j][1] - coords[i][1]) ** 2)
        mat[i][j] = round(d, 2)
        mat[j][i] = round(d, 2)

tournee_opt, dist_opt = tsp_force_brute(mat)
print(f"Tournee optimale ({nb_villes} villes) : {tournee_opt}")
print(f"Distance optimale : {dist_opt}")

# Compter le nombre de permutations evaluees
print(f"Permutations possibles : {math.factorial(nb_villes - 1)}")

## 8. Mesure experimentale de la complexite

Pour verifier experimentalement la complexite d'un algorithme, on mesure son temps d'execution pour differentes tailles d'entree et on observe la tendance.

In [ ]:
# Mesure du temps de la force brute TSP pour differentes tailles
print(f"{'n':>4} | {'Permutations':>14} | {'Temps (ms)':>12}")
print("-" * 40)

for n in range(4, 11):
    coords_test = [(random.randint(0, 100), random.randint(0, 100)) for _ in range(n)]
    mat_test = [[0.0] * n for _ in range(n)]

    for i in range(n):
        for j in range(i + 1, n):
            d = math.sqrt((coords_test[j][0] - coords_test[i][0]) ** 2 +
                          (coords_test[j][1] - coords_test[i][1]) ** 2)
            mat_test[i][j] = round(d, 2)
            mat_test[j][i] = round(d, 2)

    _, t = mesurer_temps(tsp_force_brute, mat_test)
    print(f"{n:>4} | {math.factorial(n - 1):>14} | {t:>12.1f}")

## 9. Exercices de synthese

### Exercice 1 : Fibonacci par programmation dynamique ascendante avec espace O(1)

La version iterative utilise un tableau de taille $n$. Proposez une version qui n'utilise que deux variables pour stocker les deux derniers termes.

In [ ]:
def fibonacci_optimal(n):
    """
    Fibonacci en O(n) temps et O(1) espace.

    A COMPLETER par les etudiants.
    """
    pass


# Test
# for i in range(10):
#     print(fibonacci_optimal(i), end=" ")

### Exercice 2 : Nombre de chemins dans une grille

Combien de chemins differents existent pour aller du coin haut-gauche au coin bas-droite d'une grille $m \times n$, en ne se deplacant que vers la droite ou vers le bas ? Resolvez ce probleme par programmation dynamique.

In [ ]:
def nombre_chemins_grille(m, n):
    """
    Nombre de chemins dans une grille m x n.

    A COMPLETER par les etudiants.
    """
    pass


# Test : grille 3x3 -> 6 chemins
# print(nombre_chemins_grille(3, 3))

### Exercice 3 : Comparer glouton et force brute sur le TSP

Pour une instance de 10 villes generee aleatoirement, comparez la solution du TSP glouton (plus proche voisin) avec la solution optimale (force brute). Quelle est l'ecart en pourcentage ?

In [ ]:
# A COMPLETER par les etudiants
# 1. Generer une instance de 10 villes
# 2. Calculer la matrice des distances
# 3. Resoudre avec l'heuristique gloutonne (plus proche voisin)
# 4. Resoudre par force brute
# 5. Calculer l'ecart relatif